# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank.ai_internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A: "ML model beats the hand-rule baseline on Precision@50"

**Where the label comes from:**
The label `is_declining_label` is derived from `trend_direction == 'down'`, which is itself computed by comparing impressions/clicks/sessions between the last 30 days and the previous 30 days of a 90-day window. This is a retrospective, rule-based proxy label — not a ground truth outcome (e.g., whether refreshing the page actually improved its ranking).

**Does the validation carry the claim?**
Partially. The claim "model beats the rule" holds on the internal test set (client holdout). But Precision@50 measures whether the model correctly identifies pages the *rule already labeled as declining* in the same dataset. This validates ordering-quality against a known label, not prediction of future decline. A stronger claim would require:
1. A prospective holdout: train on month T, test on month T+3 outcomes.
2. Intervention data: did pages refreshed from the queue actually recover?

**Constructive suggestion:** The finding is validly stated for this scope. Adding a note that Precision@50 is measured against a proxy label (not observed recovery) would strengthen methodological honesty.

---

### Finding B: "Content staleness (days_since_last_update) is among the top predictors of declining search performance"

**Where the label comes from:** Same proxy label as above.

**Does the validation carry the claim?**
The feature importance ranking (SHAP or split gain) confirms `days_since_last_update` is a high-importance feature in the model. However, **correlation ≠ causation**: stale content may be stale *because* its traffic was already declining and no one prioritized it — reverse causality. The finding might be re-stated as: "Within this dataset, days_since_last_update is a strong predictor of the declining proxy label" rather than asserting it causes decline.

**Constructive suggestion:** A partial dependency plot showing `days_since_last_update` vs P(declining) with confidence bands would make this finding more precise. The threshold effect (e.g., "decline probability doubles after 180 days") is more actionable than raw feature importance rank.

## 2. My model under the same scrutiny

*Apply the same methodology questions to your own model.*

**Applying the same scrutiny to the LightGBM model from w05_model.ipynb:**

**Label validity:** My model uses the same proxy label as the reference pipeline. The same limitations apply: the label captures recent retrospective trend, not future decline. A page that recovered naturally in the last 2 weeks may still be labeled declining if the earlier 4 weeks pulled the average down.

**Split design:** The client-grouped split is honest for between-client generalization, but the 32-client dataset is small enough that the 6–7 held-out clients may not represent the breadth of new clients FlyRank acquires. A more rigorous validation would repeat the split across multiple random seeds and report mean ± std of Precision@50.

**Feature leakage:** I explicitly excluded `trend_direction`, `trend_pct`, and the 30d comparison windows. I ran a correlation check in w03 and no feature exceeded 0.6 absolute correlation with the label. However, I should flag that the ratio feature `impression_drop_ratio = impressions_last_30d / impressions_prev_30d` was **not included** in the model — if it had been, it would constitute near-leakage.

**Calibration:** LightGBM's probability outputs are not guaranteed to be calibrated. For the refresh queue use case (ranking, not probability estimation), this is acceptable. If the scores were used to communicate confidence percentages to clients, I would add Platt scaling or isotonic regression.

**One thing my model cannot claim:** That refreshing the top-50 pages will definitely recover their traffic. The model only identifies pages that match the historical pattern of declining content — it cannot account for external factors (algorithm updates, competitor content, seasonality) that may have caused the decline.

In [ ]:
# Validation: Precision@K curve — how does the model's precision hold at different queue depths?
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Assume model predictions were saved from w05_model.ipynb
# If not, re-run a simplified version here

try:
    test_df = pd.read_csv('work/outputs/model_predictions.csv')
except FileNotFoundError:
    print('Run w05_model.ipynb first to generate model_predictions.csv')
    import sys; sys.exit(0)

test_df_sorted = test_df.sort_values('model_prob', ascending=False).reset_index(drop=True)

ks = list(range(10, 301, 10))
precisions = [test_df_sorted.head(k)['is_declining_label'].mean() for k in ks]
baseline_rate = test_df['is_declining_label'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ks, precisions, 'b-o', markersize=4, label='LightGBM')
ax.axhline(baseline_rate, color='red', linestyle='--', label=f'Random baseline ({baseline_rate:.1%})')
ax.set_xlabel('Queue depth K (pages reviewed)')
ax.set_ylabel('Precision@K')
ax.set_title('Precision@K — How accurate is the model at different queue depths?')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
os.makedirs('work/outputs', exist_ok=True)
plt.savefig('work/outputs/precision_at_k.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Precision@10:  {test_df_sorted.head(10)["is_declining_label"].mean():.1%}')
print(f'Precision@50:  {test_df_sorted.head(50)["is_declining_label"].mean():.1%}')
print(f'Precision@100: {test_df_sorted.head(100)["is_declining_label"].mean():.1%}')
print(f'Baseline rate: {baseline_rate:.1%}')